# Chapter 8 — Measuring similarity · Notebook to complete

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch08_a_completer.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

The cells to modify are marked **TO COMPLETE**. Run the cells in order: as long as an answer is missing, the checks say so without raising an error. The solution is in `ch08_corrige.ipynb`, on the same companion page.

**Contents.** Data · The computations of lessons 8.1 to 8.3 · Exercises 8.1 to 8.6

**Before opening Jupyter:** the exercises in this chapter are done first on paper, with a calculator. The cells below are there to **check** your calculations. As in the book, means and standard deviations divide by $n$ and are computed on the six reference sales, A to F.

**Variable names.** The code uses the book's English names: `floor_area` (floor area, in m²), `distance` (distance to the center, in km), `price` (price, in k€), `home` (name of the home).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The book's colors: reference homes in blue, selected neighbors circled in orange, distances in magenta
BLUE, ORANGE, MAGENTA = '#2a78d6', '#eb6834', '#c2185b'

## Data

The identifiers D2.1 and D2.E2 are those of the book and of the CSV files. The columns are renamed in English when the data are read; the correspondence is given in a comment.

In [ ]:
# Chapter 8 data, copied from the book's CSV files (do not modify)

# D2.1 — the seven homes; CSV columns renamed in English:
#   home = logement, floor_area = surface_m2 (m²), distance = distance_km (km), price = prix_kEUR (k€)
homes = pd.DataFrame({
    'home':       ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'floor_area': [32, 45, 54, 70, 85, 98, 62],
    'distance':   [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'price':      [118, 172, 185, 245, 279, 335, 209],
})
reference = homes[homes['home'] != 'G'].reset_index(drop=True)   # the six reference sales, A to F
X = reference[['floor_area', 'distance']].to_numpy(dtype=float)  # one row per sale: (m², km)
price = reference['price'].to_numpy(dtype=float)                 # prices of the reference sales, in k€
G = np.array([62, 2.5])                                          # home G: 62 m², 2.5 km (sold for 209 k€)

# D2.E2 — home I (exercise 8.2)
I = np.array([75, 3.5])   # 75 m², 3.5 km

# Examples from the book (lessons 8.1 and 8.3): city map in km, messages, made-up embeddings
school, G_map = np.array([4.5, 6]), np.array([1.5, 2])            # the school and G on the map, in km
R1, R2, P = np.array([3, 1]), np.array([6, 2]), np.array([1, 3])  # (“meeting”, “offer”)
embeddings = {'meeting': np.array([4, 3]), 'assembly': np.array([3, 4]),
              'discount': np.array([-3, 4]), 'promotion': np.array([-4, 3])}
# keys: the four words of table 8.3 (meeting, assembly, discount, promotion)

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `home` | home identifier (A, B…) | — | `logement` |
| `floor_area` | living area of the home | m² | `surface_m2` |
| `price` | sale price | k€ | `prix_kEUR` |
| `distance` | distance to the city center | km | `distance_km` |
| `distance_m` | distance to the city center | m | — |
| `message` | text identifier (T1 to T4) | — | — |
| `meeting_count` | number of occurrences of the word “meeting” | occurrences | — |
| `offer_count` | number of occurrences of the word “offer” | occurrences | — |
| `volunteer_count` | number of occurrences of the word “volunteer” | occurrences | — |

In [ ]:
# Provided tools: you do not need to modify them

def nearest(dists, k=3):
    """The k nearest reference sales (distances given in the order A to F) and their mean price."""
    order = np.argsort(dists, kind='stable')[:k]
    return list(reference['home'].iloc[order]), float(price[order].mean())


def show_table(dists, label):
    """Table of the distances to a home, from shortest to longest."""
    t = pd.DataFrame({'home': reference['home'], label: np.round(dists, 3), 'price': price})
    print(t.sort_values(label, kind='stable').to_string(index=False))

## The computations of the lessons, in Python

These cells redo the computations of lessons 8.1 to 8.3. Run them and compare with the book.

In [ ]:
# Lesson 8.1 — norm and distance (equation 8.1)
trip = school - G_map                       # displacement from G to the school, in km
print(f"displacement {trip}, distance {np.sqrt((trip ** 2).sum()):g} km")
print(f"from the center to G: {np.linalg.norm(G_map):g} km")             # the distance to the center in table 2.1
print(f"norm of (2, 3, 6): {np.linalg.norm([2, 3, 6]):g}")      # Pythagoras twice, or a single calculation
C_home = X[2]                                                       # C: 54 m², 2.5 km
print(f"distance from C to G: {np.linalg.norm(C_home - G):g} (mixed units: m² and km)")

In [ ]:
# Lesson 8.2 — first attempt: raw distances to G, then the distance to the center in meters
raw_G = np.sqrt(((X - G) ** 2).sum(axis=1))
show_table(raw_G, 'raw_distance')
n_km, e_km = nearest(raw_G)
print(f"3 nearest neighbors (raw, km): {n_km}, estimate {e_km:.1f} k€")
X_m, G_m = X * [1, 1000], G * [1, 1000]                             # distance to the center in meters
n_m, e_m = nearest(np.linalg.norm(X_m - G_m, axis=1))
print(f"3 nearest neighbors (raw, m):  {n_m}, estimate {e_m:.0f} k€")

In [ ]:
# Lesson 8.2 — second attempt: standardize with the reference sales (equation 8.2)
mu, sigma = X.mean(axis=0), X.std(axis=0)                           # division by n (ddof=0)
print(f"floor area: mean {mu[0]:.0f} m², standard deviation {sigma[0]:.2f} m²")
print(f"distance: mean {mu[1]:.3f} km, standard deviation {sigma[1]:.3f} km (√29/6 = {np.sqrt(29) / 6:.3f})")
Z, z_G = (X - mu) / sigma, (G - mu) / sigma
print('z of G:', np.round(z_G, 2))
std_G = np.linalg.norm(Z - z_G, axis=1)
show_table(std_G, 'std_distance')
neighbours, estimate = nearest(std_G)
print(f"3 nearest neighbors (standardized): {neighbours}, estimate {estimate:.0f} k€ (actual price: 209 k€)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(Z[:, 0], Z[:, 1], color=BLUE, s=40)
ax.scatter(*z_G, marker='D', color='black', s=40)
for h, (a, b) in zip(reference['home'], Z):
    ax.annotate(h, (a, b), xytext=(5, 4), textcoords='offset points')
ax.annotate('G', z_G, xytext=(5, 4), textcoords='offset points')
idx = np.argsort(std_G, kind='stable')[:3]
ax.scatter(Z[idx, 0], Z[idx, 1], s=220, facecolors='none', edgecolors=ORANGE, linewidths=1.8)
ax.add_patch(plt.Circle(z_G, std_G[idx[-1]], fill=False, linestyle='--', color='grey'))
ax.set_aspect('equal'); ax.set_xlabel('z floor area'); ax.set_ylabel('z distance'); ax.grid(alpha=0.4)
plt.show()

In [ ]:
# Lesson 8.3 — distance or cosine? (equation 8.3)
def cos_lesson(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"distances: R1–R2 {np.linalg.norm(R1 - R2):.2f}, R1–P {np.linalg.norm(R1 - P):.2f}")
print(f"cosines:   R1–R2 {cos_lesson(R1, R2):.2f}, R1–P {cos_lesson(R1, P):.2f}, "
      f"angle R1–P {np.degrees(np.arccos(cos_lesson(R1, P))):.0f}°")

# Preview: the embeddings of table 8.3 (all of norm 5)
for a, b in [('meeting', 'assembly'), ('discount', 'promotion'), ('meeting', 'discount'), ('meeting', 'promotion')]:
    print(f"cos({a}, {b}) = {cos_lesson(embeddings[a], embeddings[b]):.2f}")
message = (embeddings['meeting'] + embeddings['assembly']) / 2       # the message “meeting, assembly”
print(f"message {message}: cos with meeting {cos_lesson(message, embeddings['meeting']):.2f}, "
      f"with promotion {cos_lesson(message, embeddings['promotion']):.2f}")

## Exercise 8.1 — Distances on the city map

*Concepts: norm and distance (lesson 8.1, equation 8.1).* On the map in figure 8.1, in kilometers, the swimming pool is at the point $(7.5, 10)$ and the station at the point $(-3.5, 5)$; G is at the point $(1.5, 2)$.

### a) to e) On paper

*Your calculations:* …

In [ ]:
# TO COMPLETE: the norm of a vector, without np.linalg.norm (equation 8.1)
def norm(u):
    u = np.asarray(u, dtype=float)
    return None   # TO COMPLETE: square root of the sum of the squares of the coordinates

pool, station = np.array([7.5, 10]), np.array([-3.5, 5])   # swimming pool and station, in km

In [ ]:
# Check of a) to e)
if norm([6, 8]) is None:
    print('not yet completed. Complete the function norm.')
else:
    d_pool, d_station, d_station_pool = norm(pool - G_map), norm(station - G_map), norm(pool - station)
    print(f"a) {norm([6, 8]):g}   b) {d_pool:g} km   c) {d_station:.2f} km and {d_station_pool:.2f} km")
    print(f"d) detour: {d_station + d_station_pool - d_pool:.2f} km   e) {norm([2, 3, 6]):g}")
    print('→', 'correct' if abs(d_pool - 10) < 1e-9 and abs(d_station - 5.831) < 5e-4
          and abs(d_station_pool - 12.083) < 5e-4 and abs(norm([2, 3, 6]) - 7) < 1e-9 else 'check again')

## Exercise 8.2 — The neighbors of home I

*Concepts: raw distance and standardized distance (lesson 8.2, equation 8.2).* Home I (dataset D2.E2) has a `floor_area` of 75 m² and a `distance` of 3.5 km: it is the vector `I`. We estimate it with the six reference sales and $k = 3$.

- `floor_area` — floor area of the home, in m² (column `surface_m2` of the CSV);
- `distance` — distance to the center, in km (column `distance_km` of the CSV);
- `price` — prices of the reference sales, in k€; $z$-scores without a unit.

### a) to e) On paper

*Your calculations and your interpretations:* …

In [ ]:
# TO COMPLETE: distances from I to the six sales A to F, raw then standardized
raw_I = None   # TO COMPLETE: np.sqrt(((X - I) ** 2).sum(axis=1))
z_I = None     # TO COMPLETE: scores of I with mu and sigma (lesson 8.2)
std_I = None   # TO COMPLETE: distances between the rows of Z and z_I

In [ ]:
# Check of a) to c)
if raw_I is None or z_I is None or std_I is None:
    print('not yet completed. Complete the three lines.')
else:
    show_table(raw_I, 'raw_distance')
    (n_raw, e_raw), (n_std, e_std) = nearest(raw_I), nearest(std_I)
    print(f"a) raw: {n_raw}, estimate {e_raw:.1f} k€")
    print(f"b) z of I: {np.round(z_I, 2)}")
    show_table(std_I, 'std_distance')
    print(f"c) standardized: {n_std}, estimate {e_std:.0f} k€")
    print('→', 'correct' if n_raw == ['D', 'E', 'C'] and abs(e_raw - 236.33) < 0.01
          and n_std == ['E', 'C', 'A'] and abs(e_std - 194) < 1e-9 else 'check again')

## Exercise 8.3 — Meters or kilometers?

*Concepts: raw distance and units (lesson 8.2).* We go back to G and the six sales, but the distance to the center is counted in meters: `distance_m` $= 1{,}000 \times$ `distance`.

- `distance_m` — distance to the center, in m;
- `distance` — distance to the center, in km; `floor_area` (m²) and `price` (k€) of the reference sales.

### a) to d) On paper

*Your calculations and your answers:* …

In [ ]:
# TO COMPLETE: the distance to the center in meters, then the raw distances to G
distance_m = None     # TO COMPLETE: 1000 × the distance column of sales A to F
G_distance_m = None   # TO COMPLETE: distance from G to the center, in m
raw_G_m = None        # TO COMPLETE: raw distances to G with (floor_area; distance_m)

In [ ]:
# Check of a), c) and d)
if distance_m is None or G_distance_m is None or raw_G_m is None:
    print('not yet completed. Complete the three lines.')
else:
    show_table(raw_G_m, 'raw_distance_m')
    n_m, e_m = nearest(raw_G_m)
    print(f"a) in meters: {n_m}, estimate {e_m:.0f} k€")
    mu_m, sigma_m = distance_m.mean(), distance_m.std()
    z_G_m = (G_distance_m - mu_m) / sigma_m
    print(f"c) mean {mu_m:.0f} m, standard deviation {sigma_m:.0f} m; z of G: {z_G_m:.2f} in m, {z_G[1]:.2f} in km")
    print('→', 'correct' if n_m == ['C', 'E', 'A'] and abs(e_m - 194) < 1e-9 and abs(z_G_m - z_G[1]) < 1e-12
          else 'check again')

## Exercise 8.4 — How many neighbors?

*Concepts: nearest neighbors, choice of $k$ (lesson 8.2), validation (lesson 6.4).* In order of standardized distance to G, the six sales and their `price` are C (185 k€), E (279), B (172), A (118), D (245) and F (335).

- `price` — sale price, in k€; $k$ — number of neighbors kept, without a unit.

### a) to c) On paper

*Your estimates and your answer:* …

In [ ]:
# TO COMPLETE: the estimate for G for each value of k
estimates = None   # TO COMPLETE: {k: nearest(std_G, k)[1] for k in (1, 2, 3, 4, 6)}

In [ ]:
# Check of a) and c)
if estimates is None:
    print('not yet completed. Complete the line.')
else:
    for k, e in estimates.items():
        print(f"k = {k}: {e:.1f} k€, difference from the actual price (209 k€): {e - 209:+.1f} k€")
    expected = {1: 185, 2: 232, 3: 212, 4: 188.5, 6: 1334 / 6}
    print('→', 'correct' if all(abs(estimates[k] - v) < 1e-9 for k, v in expected.items()) else 'check again')

## Exercise 8.5 — Distance or cosine?

*Concepts: distance (equation 8.1), cosine (equation 8.3).* We count three words in four of the association's messages: “meeting”, “offer” and “volunteer” (table 8.4 in the book).

- `message` — message identifier, T1 to T4;
- `meeting_count`, `offer_count`, `volunteer_count` — number of times the words “meeting”, “offer” and “volunteer” are used in the message: the three coordinates of its vector.

In [ ]:
# Data for exercise 8.5 (table in the book)
words = pd.DataFrame({
    'message':         ['T1', 'T2', 'T3', 'T4'],
    'meeting_count':   [2, 6, 1, 0],
    'offer_count':     [0, 0, 2, 1],
    'volunteer_count': [1, 3, 2, 0],
}).set_index('message')
T = {m: words.loc[m].to_numpy(dtype=float) for m in words.index}
print(words)

### a) to d) On paper

*Your calculations and your interpretation:* …

In [ ]:
# TO COMPLETE: the cosine of two vectors (equation 8.3)
def cosine(u, v):
    return None   # TO COMPLETE: dot product divided by the product of the norms

In [ ]:
# Check of a) to c)
if cosine(T['T1'], T['T2']) is None:
    print('not yet completed. Complete the function cosine.')
else:
    results = pd.DataFrame({
        'norm': [np.linalg.norm(T[m]) for m in T],
        'distance_to_T1': [np.linalg.norm(T['T1'] - T[m]) for m in T],
        'cosine_with_T1': [cosine(T['T1'], T[m]) for m in T],
    }, index=list(T)).round(2)
    print(results)
    print('→', 'correct' if abs(cosine(T['T1'], T['T2']) - 1) < 1e-12 and abs(cosine(T['T1'], T['T3']) - 0.596) < 5e-4
          and abs(cosine(T['T1'], T['T4'])) < 1e-12 else 'check again')

## Exercise 8.6 — Challenge: normalize, then compare

*Concepts: norm and dot product (equation 8.1), cosine (equation 8.3).* Questions a) and b) are proofs on paper; the notebook checks c) and d).

### a) to d) On paper

*Your proofs and your calculations:* …

In [ ]:
# TO COMPLETE: each message divided by its norm, then the distances to the normalized T1
normalized = None     # TO COMPLETE: {m: T[m] / np.linalg.norm(T[m]) for m in T}
unit_dist = None      # TO COMPLETE: {m: distance between normalized['T1'] and normalized[m] for m in T}

In [ ]:
# Check of c) and d)
if normalized is None or unit_dist is None:
    print('not yet completed. Complete the two lines.')
else:
    for m in ('T2', 'T3', 'T4'):
        c = float(np.dot(T['T1'], T[m]) / (np.linalg.norm(T['T1']) * np.linalg.norm(T[m])))
        print(f"{m}: normalized distance {unit_dist[m]:.2f}, √(2 − 2 cos) = {np.sqrt(max(2 - 2 * c, 0)):.2f}")
    u, v = embeddings['meeting'] / 5, embeddings['assembly'] / 5
    print(f"d) meeting and assembly: directly {np.linalg.norm(u - v):.2f}, with b) {np.sqrt(2 - 2 * np.dot(u, v)):.2f}")
    print('→', 'correct' if abs(unit_dist['T2']) < 1e-9 and abs(unit_dist['T3'] - 0.899) < 5e-4
          and abs(unit_dist['T4'] - np.sqrt(2)) < 1e-9 else 'check again')